In [ ]:
# NOTEBOOK CELLS for Akili Robotics v0.2 MuJoCo — assembled into .ipynb after verification
# marker format:  # %% [md]  or  # %%

# AKILI ROBOTICS v0.2 — *"Four systems, one survivor, then beyond"*
**Standalone notebook — MuJoCo 3D arm, CPU-only (Colab or a Mac).**
#
A 3-DOF robot arm in real 3D-rendered physics learns skills sequentially:
reach two targets, press a button, and route around a hazard column.
Then a **poisoned skill** (inverted joint commands — the flipped-actuator nightmare)
and a **poisoned skill update** arrive. Four systems face the same world, same data, same poison:
#
1. **Sequential fine-tune** — one shared model: forgets old skills on camera, poison baked in
2. **Motion-library retrieval** (the robotics RAG) — replays recordings, dies on a 5 cm shift
3. **Naive adapter bank** — modularity without governance: ships the poison to production
4. **Akili** — frozen trunk + write-once adapters + validation gates + versioned rollback
#
Then Akili goes *beyond* survival:
- **Act 4** — a poisoned *skill update* (v2 drops the hazard routing) is shadow-tested and rejected while v1 keeps working: zero downtime
- **Act 5** — certified skill *composition*: a chained mission with a cryptographic receipt per step
#
**Runtime:** ~15–25 min on CPU. Fully resumable (`AKILI_ARM_RESUME=latest`).
Everything (checkpoints, registry, audit chain, report, GIFs) lands in one run folder.
Control is kinematic in MuJoCo (full 3D collision geometry + 3D rendering); training is CPU torch.

In [ ]:
# ============================================================
# CELL 1 — CONFIG
# ============================================================
import os, json

def _env(n, d):
    v = os.environ.get(n, "")
    return v if str(v).strip() else d

CONFIG = {
    "seed": int(_env("AKILI_ARM_SEED", "1")),
    "out_dir": _env("AKILI_ARM_OUT", "/content/drive/MyDrive/AKM_CLR/stage05/akili_robotics_v0_2_mujoco" if
                    os.path.isdir("/content/drive") else os.path.expanduser("~/akili_robotics_v0_2")),
    "resume": _env("AKILI_ARM_RESUME", ""),
    "steps": 100, "dt": 0.05, "success_dist": 0.025, "qdot_max": 1.2,
    "hidden": 128, "adapter_rank": 8,
    "train": {"epochs": 70, "lr": 2e-3, "batch": 256, "eps_per_skill": 300},
    "eval_episodes": 50,
    "activation_min_success": 0.85,
    "skills": ["reach_A", "reach_B", "press_button", "avoid_zone"],
    "poison_skill": "reach_C",
    "update_skill": "avoid_zone",     # Act 4: the update that drops hazard routing
    "v2_poison_frac": 0.35,          # fraction of corner-cutting trajectories in the v2 candidate
}
print(json.dumps(CONFIG, indent=2))

In [ ]:
# ============================================================
# CELL 2 — IMPORTS, GL SETUP, DETERMINISM, RUN DIR
# ============================================================
import os, json, glob, math, time, hashlib, datetime, random
import numpy as np

# --- dependency guard (Colab: mujoco is not preinstalled) ---
import sys, subprocess
try:
    import mujoco  # noqa
except ImportError:
    print("[env] installing mujoco, imageio, pillow...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mujoco", "imageio", "pillow"],
                   check=False)

# --- GL backend selection MUST happen before importing mujoco ---
# order: macOS/Windows native -> DISPLAY (Linux desktop) -> EGL (Colab/headless) -> osmesa
import ctypes.util
GL_MODE = None
if sys.platform == "darwin" or sys.platform.startswith("win"):
    GL_MODE = "native"                 # mujoco uses CGL/WGL offscreen contexts
elif os.environ.get("DISPLAY"):
    GL_MODE = "glfw"
elif ctypes.util.find_library("EGL"):
    os.environ["MUJOCO_GL"] = "egl"; GL_MODE = "egl"
elif ctypes.util.find_library("OSMesa"):
    os.environ["MUJOCO_GL"] = "osmesa"; GL_MODE = "osmesa"

import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

MUJOCO_OK = True
try:
    import mujoco
    import imageio.v2 as imageio
except Exception as e:
    MUJOCO_OK = False
    print("[env] mujoco unavailable:", e)

SEED = CONFIG["seed"]
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
print(f"[env] torch={torch.__version__} mujoco={mujoco.__version__ if MUJOCO_OK else 'n/a'} GL={GL_MODE}")

def resolve_out():
    if CONFIG["resume"] and CONFIG["resume"] != "latest":
        assert os.path.isdir(CONFIG["resume"])
        return CONFIG["resume"]
    if CONFIG["resume"] == "latest":
        runs = sorted(glob.glob(os.path.join(CONFIG["out_dir"], "run_*")))
        assert runs, f"[resume] no runs under {CONFIG['out_dir']}"
        return runs[-1]
    return os.path.join(CONFIG["out_dir"],
                        "run_" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ"))

RUN_DIR = resolve_out()
os.makedirs(RUN_DIR, exist_ok=True)
CKPT = os.path.join(RUN_DIR, "checkpoints"); os.makedirs(CKPT, exist_ok=True)
ANIM = os.path.join(RUN_DIR, "animations"); os.makedirs(ANIM, exist_ok=True)
print(f"[run] {RUN_DIR}")

In [ ]:
# ============================================================
# CELL 3 — THE WORLD: 3-DOF SCARA ARM, HAZARD COLUMN, TARGETS
# ============================================================
SCENE_XML = """
<mujoco model="akili_arm">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0.3 -0.3 1.4" dir="-0.15 0.25 -1" diffuse="0.9 0.9 0.9" specular="0.25 0.25 0.25"/>
    <geom name="table" type="plane" pos="0 0 0" size="1.2 1.2 0.1" rgba="0.86 0.88 0.91 1"/>
    <geom name="hazard" type="cylinder" pos="0.30 0.08 0.10" size="0.07 0.10" rgba="0.88 0.10 0.10 0.92"/>
    <geom name="button" type="cylinder" pos="0.10 -0.25 0.015" size="0.035 0.015" rgba="1.0 0.78 0.10 1"/>
    <body name="base" pos="-0.12 -0.08 0.03">
      <geom type="cylinder" size="0.05 0.03" rgba="0.22 0.22 0.28 1"/>
      <body name="link1" pos="0 0 0.035">
        <joint name="j1" type="hinge" axis="0 0 1" range="-2.7 2.7" damping="0.02"/>
        <geom type="capsule" fromto="0 0 0 0.25 0 0" size="0.030" rgba="0.16 0.38 0.80 1"/>
        <body name="link2" pos="0.25 0 0">
          <joint name="j2" type="hinge" axis="0 0 1" range="-2.7 2.7" damping="0.02"/>
          <geom type="capsule" fromto="0 0 0 0.20 0 0" size="0.028" rgba="0.20 0.48 0.86 1"/>
          <body name="link3" pos="0.20 0 0">
            <joint name="j3" type="hinge" axis="0 0 1" range="-2.7 2.7" damping="0.02"/>
            <geom type="capsule" fromto="0 0 0 0.15 0 0" size="0.026" rgba="0.30 0.58 0.92 1"/>
            <geom name="eeball" type="sphere" pos="0.15 0 0" size="0.024" rgba="0.95 0.25 0.20 1"/>
            <site name="ee" pos="0.15 0 0" size="0.006"/>
          </body>
        </body>
      </body>
    </body>
    <body name="goal_marker" mocap="true" pos="0 0 0.02">
      <geom type="sphere" size="0.022" rgba="0.10 0.75 0.25 0.9" contype="0" conaffinity="0"/>
    </body>
    <body name="wp_marker" mocap="true" pos="0 0 0.02">
      <geom type="sphere" size="0.014" rgba="0.10 0.75 0.90 0.9" contype="0" conaffinity="0"/>
    </body>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(SCENE_XML)
data = mujoco.MjData(model)
scratch = mujoco.MjData(model)          # private state for expert/FK — never touches live data

EE_SITE = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "ee")
LINK2_ID = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "link2")
LINK3_ID = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "link3")
GOAL_MID = int(model.body_mocapid[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "goal_marker")])
WP_MID = int(model.body_mocapid[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "wp_marker")])

BASE = np.array([-0.12, -0.08])
HAZARD = {"c": np.array([0.30, 0.08]), "r": 0.07, "margin": 0.04}     # margin ~ capsule radii
RM = HAZARD["r"] + HAZARD["margin"]                                   # 0.11 no-contact radius
TARGETS = {"reach_A": np.array([-0.30, 0.28]), "reach_B": np.array([0.33, 0.26]),
           "reach_C": np.array([-0.05, 0.45])}
BUTTON = np.array([0.10, -0.25])
AVOID_GOAL = np.array([0.34, -0.22])
AVOID_WP = np.array([0.006, -0.09])                                   # certified dogleg corner
PRESS_WP = np.array([0.0, -0.15])                                     # press approach corner (keeps the sweep clear)
Q_HOME = np.array([0.8, 0.8, -0.7])
VMAX_EE, QDOT_MAX, LAM = 0.22, CONFIG["qdot_max"], 0.05
CLEAR = 0.05
REACT_LINK, GAIN_LINK = RM + 0.15, 6.0

def reset_to(q):
    mujoco.mj_resetData(model, data)
    data.qpos[:] = q; data.qvel[:] = 0.0
    mujoco.mj_forward(model, data)

def ee_pos(d=None):
    d = data if d is None else d
    return d.site_xpos[EE_SITE][:2].copy()

def joint_xy(d=None):
    d = data if d is None else d
    return [BASE.copy(), d.xpos[LINK2_ID][:2].copy(), d.xpos[LINK3_ID][:2].copy(), ee_pos(d)]

def hazard_clearance():
    """min distance from any link segment to hazard center, minus the no-contact radius. >0 = safe."""
    pts = joint_xy(); best = 1e9
    for i in range(3):
        a, b = pts[i], pts[i + 1]; ab = b - a
        s = float(np.clip(((HAZARD["c"] - a) @ ab) / (ab @ ab + 1e-12), 0, 1))
        best = min(best, float(np.linalg.norm(a + s * ab - HAZARD["c"])))
    return best - RM

print("[world] 3-DOF arm ready | skills:", CONFIG["skills"], "| poison:", CONFIG["poison_skill"],
      "| hazard no-contact radius:", RM)

In [ ]:
# ============================================================
# CELL 4 — EXPERT + CLOSED-LOOP ROLLOUT (kinematic control, full collision checks)
# ============================================================
def _los_blocked(p1, p2, r_eff):
    c = HAZARD["c"]; v = p2 - p1; L2 = float(v @ v)
    if L2 < 1e-12: return bool(np.linalg.norm(p1 - c) < r_eff)
    t = float(np.clip(((c - p1) @ v) / L2, 0.0, 1.0))
    return bool(np.linalg.norm(p1 + t * v - c) < r_eff)

def _point_jacobians():
    Js = [np.zeros((2, 3))]
    for bid in (LINK2_ID, LINK3_ID):
        jb = np.zeros((3, model.nv)); jr = np.zeros((3, model.nv))
        mujoco.mj_jacBody(model, scratch, jb, jr, bid)
        Js.append(jb[:2, :].copy())
    jacp = np.zeros((3, model.nv)); jacr = np.zeros((3, model.nv))
    mujoco.mj_jacSite(model, scratch, jacp, jacr, EE_SITE)
    Js.append(jacp[:2, :].copy())
    return Js

def expert_action(q, goal, skill, degraded=False):
    """Jacobian expert. avoid_zone routes through a certified waypoint (never hugs the cylinder);
    every skill gets whole-arm nullspace hazard repulsion. `degraded` = the lazy straight-line
    expert used ONLY to manufacture the poisoned v2 update data."""
    scratch.qpos[:] = q; scratch.qvel[:] = 0.0
    mujoco.mj_forward(model, scratch)
    ee = ee_pos(scratch)
    tgt = goal
    if skill == "avoid_zone" and not degraded:
        if _los_blocked(ee, goal, RM + CLEAR) and np.linalg.norm(ee - AVOID_WP) > 0.06:
            tgt = AVOID_WP
    elif skill == "press_button" and not degraded:
        if float(np.dot(ee - PRESS_WP, goal - PRESS_WP)) < 0:   # approach corner not passed yet
            tgt = PRESS_WP
    d = tgt - ee
    n = np.linalg.norm(d) + 1e-9
    if skill == "press_button" and np.linalg.norm(ee - goal) < 0.02:
        v = np.zeros(2)
    else:
        v = d / n * min(VMAX_EE, n)
    Js = _point_jacobians()
    J = Js[-1]
    Jinv = J.T @ np.linalg.inv(J @ J.T + LAM**2 * np.eye(2))
    sec = 0.25 * (Q_HOME - q)
    if not degraded:
        pts = joint_xy(scratch)
        for i in range(3):
            a, b = pts[i], pts[i + 1]; ab = b - a
            s = float(np.clip(((HAZARD["c"] - a) @ ab) / (ab @ ab + 1e-12), 0, 1))
            pstar = a + s * ab
            dvec = pstar - HAZARD["c"]; dist = np.linalg.norm(dvec) + 1e-9
            if dist < REACT_LINK:
                Jstar = (1 - s) * Js[i] + s * Js[i + 1]
                sec = sec + Jstar.T @ (dvec / dist * (REACT_LINK - dist) * GAIN_LINK)
    qdot = Jinv @ v + (np.eye(3) - Jinv @ J) @ sec
    return np.clip(qdot, -QDOT_MAX, QDOT_MAX)

def rollout(policy, q0, goal, skill, steps=None, record=False):
    """Closed loop: policy(q, goal, t) -> joint velocities; q integrates kinematically;
    collision geometry checked every step on the true link segments."""
    steps = steps or CONFIG["steps"]
    q = np.array(q0, dtype=float)
    reset_to(q)
    min_cl = hazard_clearance()
    qs = [q.copy()]
    for t in range(steps):
        a = np.clip(policy(q.copy(), goal, t), -QDOT_MAX, QDOT_MAX)
        q = np.clip(q + a * CONFIG["dt"], -2.7, 2.7)
        reset_to(q)
        min_cl = min(min_cl, hazard_clearance())
        if record: qs.append(q.copy())
    final = float(np.linalg.norm(ee_pos() - goal))
    success = final < CONFIG["success_dist"]
    res = {"success": success, "final": final, "clearance": min_cl,
           "safe": min_cl > 0.0}
    if skill == "avoid_zone":
        success = success and min_cl > 0.0
        res["success"] = success
    if record: res["qs"] = qs
    return res

print("[expert] jacobian expert + waypoint routing + whole-arm repulsion ready")

In [ ]:
# ============================================================
# CELL 5 — DATA GENERATION (clean skills + inverted poison + degraded v2 update)
# ============================================================
def goal_of(skill, rng):
    if skill in TARGETS: g = TARGETS[skill] + rng.uniform(-0.015, 0.015, 2)
    elif skill == "press_button": g = BUTTON + rng.uniform(-0.010, 0.010, 2)
    elif skill == "avoid_zone": g = AVOID_GOAL + rng.uniform(-0.020, 0.020, 2)
    return g

def obs_of(q, goal):
    scratch.qpos[:] = q; mujoco.mj_forward(model, scratch)
    return np.concatenate([q, ee_pos(scratch), goal]).astype(np.float32)

def gen_episode(skill, rng, poisoned=False, degraded=False):
    q0 = Q_HOME + rng.normal(0, 0.05, 3)
    goal = goal_of(skill, rng)
    q = q0.copy(); reset_to(q)
    O, A = [], []
    for t in range(CONFIG["steps"]):
        a = expert_action(q, goal, skill, degraded=degraded)
        if poisoned:
            a = -a                                   # INVERTED JOINT COMMANDS
        a = np.clip(a + rng.normal(0, 0.01, 3), -QDOT_MAX, QDOT_MAX)
        O.append(obs_of(q, goal)); A.append(a)
        q = np.clip(q + a * CONFIG["dt"], -2.7, 2.7)
        reset_to(q)
    return np.array(O), np.array(A)

def build_dataset(tag, skill, n, poisoned=False, degraded_frac=0.0):
    seed = SEED * 991 + int(hashlib.sha256(tag.encode()).hexdigest(), 16) % 997
    rng = np.random.default_rng(seed)
    O, A = [], []
    for i in range(n):
        degraded = rng.uniform() < degraded_frac
        o, a = gen_episode(skill, rng, poisoned=poisoned, degraded=degraded)
        O.append(o); A.append(a)
    return torch.tensor(np.concatenate(O)).float(), torch.tensor(np.concatenate(A)).float()

DATA = {s: build_dataset(s, s, CONFIG["train"]["eps_per_skill"]) for s in CONFIG["skills"]}
DATA[CONFIG["poison_skill"]] = build_dataset(CONFIG["poison_skill"], CONFIG["poison_skill"],
                                             CONFIG["train"]["eps_per_skill"], poisoned=True)
# the v2 update candidate: mostly clean, spiked with hazard-clipping demonstrations
V2_TAG = f"{CONFIG['update_skill']}@v2"
DATA[V2_TAG] = build_dataset(V2_TAG, CONFIG["update_skill"], CONFIG["train"]["eps_per_skill"],
                             degraded_frac=CONFIG["v2_poison_frac"])
DATA_HASHES = {s: hashlib.sha256(o.numpy().tobytes()).hexdigest() for s, (o, a) in DATA.items()}
print("[data]", {s: tuple(o.shape) for s, (o, a) in DATA.items()})

In [ ]:
# ============================================================
# CELL 6 — POLICY MODELS (frozen trunk + write-once low-rank adapters)
# ============================================================
class Trunk(nn.Module):
    def __init__(self, in_dim=7, hid=CONFIG["hidden"], out_dim=3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hid), nn.ReLU(), nn.Linear(hid, hid), nn.ReLU())
        self.head = nn.Linear(hid, out_dim)
    def features(self, x): return self.net(x)
    def forward(self, x): return self.head(self.net(x))

class AdapterPolicy(nn.Module):
    """Frozen trunk + one write-once low-rank residual adapter per skill."""
    def __init__(self, trunk):
        super().__init__()
        self.trunk = trunk
        for p in self.trunk.parameters(): p.requires_grad_(False)
        hid = CONFIG["hidden"]; r = CONFIG["adapter_rank"]
        self.A = nn.Linear(hid, r, bias=False); self.B = nn.Linear(r, hid, bias=False)
        nn.init.zeros_(self.B.weight)
    def forward(self, x):
        f = self.trunk.features(x)
        return self.trunk.head(f + self.B(self.A(f)))

def policy_fn(pol):
    pol.eval()
    def f(q, goal, t):
        with torch.no_grad():
            x = torch.tensor(obs_of(q, goal)).unsqueeze(0)
            return pol(x)[0].numpy()
    return f

def train_model(mdl, obs, acts, epochs, lr, batch, tag):
    mdl.train()
    opt = torch.optim.Adam([p for p in mdl.parameters() if p.requires_grad], lr=lr)
    n = len(obs); g = torch.Generator().manual_seed(SEED)
    for ep in range(epochs):
        perm = torch.randperm(n, generator=g)
        tot = 0.0
        for i in range(0, n, batch):
            idx = perm[i:i + batch]
            loss = nn.functional.mse_loss(mdl(obs[idx]), acts[idx])
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
        if (ep + 1) % max(1, epochs // 3) == 0:
            print(f"[train] {tag} epoch {ep+1}/{epochs} loss={tot/(n//batch+1):.5f}")
    mdl.eval()
    return mdl

def fp_of(mdl):
    h = hashlib.sha256()
    with torch.no_grad():
        for p in mdl.parameters():
            h.update(p.detach().numpy().tobytes())
    return h.hexdigest()

print("[model] trunk + low-rank adapter policy ready")

In [ ]:
# ============================================================
# CELL 7 — AKILI RUNTIME CORE (versioned registry + hash-chained audit + renderer)
# ============================================================
class AuditLog:
    def __init__(self, path):
        self.path = path
        self.entries = json.load(open(path)) if os.path.exists(path) else []
    def log(self, op, skill, details):
        prev = self.entries[-1]["hash"] if self.entries else "GENESIS"
        payload = json.dumps({"ts": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                              "op": op, "skill": skill, "details": details, "prev": prev}, sort_keys=True)
        h = hashlib.sha256(payload.encode()).hexdigest()
        self.entries.append(json.loads(payload) | {"hash": h})
        json.dump(self.entries, open(self.path, "w"), indent=2)
    def verify_chain(self):
        prev = "GENESIS"
        for e in self.entries:
            payload = json.dumps({k: e[k] for k in ("ts", "op", "skill", "details", "prev")}, sort_keys=True)
            if e["prev"] != prev or hashlib.sha256(payload.encode()).hexdigest() != e["hash"]:
                return False
            prev = e["hash"]
        return True

class Registry:
    """Version-aware: rows are skill@version; an active pointer per skill family."""
    def __init__(self, path, audit):
        self.path, self.audit = path, audit
        self.skills = json.load(open(path))["skills"] if os.path.exists(path) else {}
    def _save(self): json.dump({"skills": self.skills}, open(self.path, "w"), indent=2)
    def add(self, name, ckpt_path, data_hash, version=1, parent=None):
        key = f"{name}@v{version}"
        assert key not in self.skills, f"{key} exists (write-once)"
        self.skills[key] = {"state": "REGISTERED", "ckpt": ckpt_path, "data_hash": data_hash,
                            "validation": None, "version": version, "parent": parent,
                            "history": ["REGISTERED"]}
        self._save(); self.audit.log("ADD", key, {})
        return key
    def record_validation(self, key, card):
        assert self.skills[key]["state"] in ("REGISTERED", "VALIDATED")
        self.skills[key]["validation"] = card; self.skills[key]["state"] = "VALIDATED"
        self.skills[key]["history"].append("VALIDATED")
        self._save()
        self.audit.log("VALIDATE", key, {"success": card["success_rate"], "safe": card["safe"]})
    def activate(self, key):
        s = self.skills[key]
        assert s["state"] == "VALIDATED"
        assert s["validation"]["success_rate"] >= CONFIG["activation_min_success"], (
            f"{key} success {s['validation']['success_rate']:.2f} below minimum")
        assert s["validation"]["safe"], f"{key} failed safety check"
        s["state"] = "ACTIVE"; s["history"].append("ACTIVE")
        self._save(); self.audit.log("ACTIVATE", key, {})
    def rollback(self, key, reason=""):
        assert self.skills[key]["state"] in ("ACTIVE", "VALIDATED", "QUARANTINED")
        self.skills[key]["state"] = "ROLLED_BACK"
        self.skills[key]["history"].append(f"ROLLED_BACK({reason})")
        self._save(); self.audit.log("ROLLBACK", key, {"reason": reason})
    def active(self): return [k for k, s in self.skills.items() if s["state"] == "ACTIVE"]
    def active_version(self, name):
        vs = [(s["version"], k) for k, s in self.skills.items()
              if k.split("@v")[0] == name and s["state"] == "ACTIVE"]
        return max(vs)[1] if vs else None
    def require_executable(self, key):
        s = self.skills.get(key)
        assert s and s["state"] == "ACTIVE", f"skill {key} not executable (state={s and s['state']})"

AUDIT = AuditLog(os.path.join(RUN_DIR, "audit_log.json"))
REGISTRY = Registry(os.path.join(RUN_DIR, "registry.json"), AUDIT)

# --- renderer + captioned GIF writer (PIL captions => frames unique, video self-explanatory)
from PIL import Image, ImageDraw

RENDER_OK = GL_MODE is not None
renderer = None
if RENDER_OK:
    try:
        renderer = mujoco.Renderer(model, height=480, width=640)
        cam = mujoco.MjvCamera()
        cam.lookat = np.array([0.10, 0.05, 0.0]); cam.distance = 1.05
        cam.azimuth = 90; cam.elevation = -50
    except Exception as e:
        RENDER_OK = False
        print("[render] unavailable, GIFs skipped:", e)

def render_frame(q, goal=None, wp=None, caption=("", "")):
    reset_to(q)
    if goal is not None: data.mocap_pos[GOAL_MID] = [goal[0], goal[1], 0.02]
    if wp is not None: data.mocap_pos[WP_MID] = [wp[0], wp[1], 0.02]
    mujoco.mj_forward(model, data)
    renderer.update_scene(data, camera=cam)
    img = renderer.render().copy()
    pil = Image.fromarray(img)
    dr = ImageDraw.Draw(pil)
    top, bottom = caption
    dr.rectangle([0, 0, 640, 26], fill=(15, 15, 20))
    dr.rectangle([0, 454, 640, 480], fill=(15, 15, 20))
    dr.text((8, 6), top, fill=(240, 240, 240))
    dr.text((8, 460), bottom, fill=(180, 220, 255))
    return np.array(pil)

def make_gif(frames, fname, fps=12):
    out = os.path.join(ANIM, fname)
    imageio.mimsave(out, frames, fps=fps, loop=0)
    return out

print("[runtime] registry + audit + renderer ready | render:", RENDER_OK)

In [ ]:
# ============================================================
# CELL 8 — TRAIN TRUNK (frozen after) + SKILL ADAPTERS (write-once, resume-safe)
# ============================================================
TRUNK_PATH = os.path.join(CKPT, "trunk.pt")

if os.path.exists(TRUNK_PATH):
    trunk = Trunk(); trunk.load_state_dict(torch.load(TRUNK_PATH))
    print("[resume] trunk loaded")
else:
    print("[train] trunk on mixed data (all good skills)...")
    O = torch.cat([DATA[s][0] for s in CONFIG["skills"]])
    A = torch.cat([DATA[s][1] for s in CONFIG["skills"]])
    trunk = train_model(Trunk(), O, A, CONFIG["train"]["epochs"], CONFIG["train"]["lr"],
                        CONFIG["train"]["batch"], "trunk")
    torch.save(trunk.state_dict(), TRUNK_PATH)
for p in trunk.parameters(): p.requires_grad_(False)
TRUNK_FP = fp_of(trunk)
print(f"[trunk] frozen | fingerprint={TRUNK_FP[:16]}...")

def adapter_ckpt(key): return os.path.join(CKPT, f"adapter_{key.replace('@', '_')}.pt")

def train_adapter(key, data_key):
    path = adapter_ckpt(key)
    if os.path.exists(path):
        print(f"[resume] {key} adapter intact — skipping")
        return path
    mdl = AdapterPolicy(trunk)
    mdl = train_model(mdl, DATA[data_key][0], DATA[data_key][1], CONFIG["train"]["epochs"],
                      CONFIG["train"]["lr"], CONFIG["train"]["batch"], f"adapter:{key}")
    torch.save({"A": mdl.A.state_dict(), "B": mdl.B.state_dict()}, path)
    return path

def load_adapter_model(key):
    mdl = AdapterPolicy(trunk)
    sd = torch.load(adapter_ckpt(key))
    mdl.A.load_state_dict(sd["A"]); mdl.B.load_state_dict(sd["B"])
    mdl.eval()
    return mdl

def skill_of(key): return key.split("@v")[0]

for s in CONFIG["skills"]:
    key = f"{s}@v1"
    p = train_adapter(key, s)
    if key not in REGISTRY.skills:
        REGISTRY.add(s, p, DATA_HASHES[s], version=1)
    print(f"[bank] {key} state={REGISTRY.skills[key]['state']}")

p = CONFIG["poison_skill"]
pk = f"{p}@v1"
path = train_adapter(pk, p)
if pk not in REGISTRY.skills:
    REGISTRY.add(p, path, DATA_HASHES[p], version=1)
print(f"[bank] {pk} state={REGISTRY.skills[pk]['state']}  (candidate — not yet validated)")

path = train_adapter(V2_TAG, V2_TAG)
if V2_TAG not in REGISTRY.skills:
    REGISTRY.add(CONFIG["update_skill"], path, DATA_HASHES[V2_TAG], version=2,
                 parent=f"{CONFIG['update_skill']}@v1")
print(f"[bank] {V2_TAG} state={REGISTRY.skills[V2_TAG]['state']}  (update candidate)")

In [ ]:
# ============================================================
# CELL 9 — VALIDATION + ACTIVATION (closed-loop success AND safety, shadow eval)
# ============================================================
def validate_skill(key):
    skill = skill_of(key)
    pf = policy_fn(load_adapter_model(key))
    rng = np.random.default_rng(SEED * 313 + int(hashlib.sha256(key.encode()).hexdigest(), 16) % 997)
    res = []
    for _ in range(CONFIG["eval_episodes"]):
        q0 = Q_HOME + rng.normal(0, 0.05, 3)
        res.append(rollout(pf, q0, goal_of(skill, rng), skill))
    sr = float(np.mean([r["success"] for r in res]))
    safe = all(r["safe"] for r in res)
    card = {"skill": key, "episodes": len(res), "success_rate": sr, "safe": safe,
            "mean_final": float(np.mean([r["final"] for r in res])),
            "min_hazard_clearance": float(min(r["clearance"] for r in res)),
            "validated_at": datetime.datetime.now(datetime.timezone.utc).isoformat()}
    card["card_hash"] = hashlib.sha256(
        json.dumps({k: v for k, v in card.items() if k != "card_hash"}, sort_keys=True).encode()).hexdigest()
    return card

print("[akili] validating good skills (50 closed-loop episodes each)...")
for s in CONFIG["skills"]:
    key = f"{s}@v1"
    if REGISTRY.skills[key]["state"] == "ACTIVE":
        print(f"[resume] {key} ACTIVE")
        continue
    card = validate_skill(key)
    REGISTRY.record_validation(key, card)
    REGISTRY.activate(key)
    print(f"[validate] {key}: success={card['success_rate']:.2f} safe={card['safe']} "
          f"clearance={card['min_hazard_clearance']:.3f} -> ACTIVE")

print("\n[akili] poisoned candidate enters validation...")
pk = f"{CONFIG['poison_skill']}@v1"
if REGISTRY.skills[pk]["state"] == "REGISTERED":
    card = validate_skill(pk)
    REGISTRY.record_validation(pk, card)
    print(f"[validate] {pk}: success={card['success_rate']:.2f} safe={card['safe']}")
    try:
        REGISTRY.activate(pk)
        print("[warn] poison ACTIVATED — gates failed!")
    except AssertionError as e:
        print(f"[gate] ACTIVATION BLOCKED: {e}")
        REGISTRY.rollback(pk, "failed validation: inverted-control training data")
        print(f"[lifecycle] {pk} = ROLLED_BACK | active: {REGISTRY.active()}")

In [ ]:
# ============================================================
# CELL 10 — ROUTER (goal space, ACTIVE-only) + EXECUTION GUARD
# ============================================================
SKILL_VEC = {s: goal_of(s, np.random.default_rng(0)) for s in CONFIG["skills"] + [CONFIG["poison_skill"]]}

def route(goal):
    """Nearest ACTIVE skill anchor in goal space. Rolled-back skills are invisible."""
    act = REGISTRY.active()
    if not act: return None
    g = np.asarray(goal, dtype=float)
    d = {k: float(np.linalg.norm(g - SKILL_VEC[skill_of(k)])) for k in act}
    return min(d, key=d.get)

def execute(key, q0, goal, record=False):
    REGISTRY.require_executable(key)
    skill = skill_of(key)
    return rollout(policy_fn(load_adapter_model(key)), q0, goal, skill, record=record)

print("[router] ACTIVE-only routing ready | active:", REGISTRY.active())

In [ ]:
# ============================================================
# CELL 11 — ACT 1: THE FOUR SKILLS (rendered)
# ============================================================
act1_path = os.path.join(ANIM, "act1_skills.gif")
if RENDER_OK and not os.path.exists(act1_path):
    rng = np.random.default_rng(7)
    frames = []
    for key in REGISTRY.active():
        skill = skill_of(key)
        goal = goal_of(skill, rng)
        r = execute(key, Q_HOME + rng.normal(0, 0.05, 3), goal, record=True)
        wp = {"avoid_zone": AVOID_WP, "press_button": PRESS_WP}.get(skill)
        for i, q in enumerate(r["qs"][::4]):
            frames.append(render_frame(q, goal, wp,
                (f"AKILI | {skill} | t={i*4:03d}", f"step {i*4:03d}/{len(r['qs'])-1} | certified skill executing")))
    act1 = make_gif(frames, "act1_skills.gif")
    print(f"[anim] act1: {len(frames)} frames -> {act1}")
else:
    print(f"[anim] act1 skipped (render={RENDER_OK} or exists)")

In [ ]:
# ============================================================
# CELL 12 — SYSTEM 1: SEQUENTIAL FINE-TUNE (forgetting + baked-in poison)
# ============================================================
print("=" * 92)
print("SYSTEM 1 — sequential fine-tune of a single shared policy")
print("=" * 92)

def eval_model_on(mdl, skill, n=30):
    pf = policy_fn(mdl)
    rng = np.random.default_rng(SEED + 11)
    return float(np.mean([rollout(pf, Q_HOME + rng.normal(0, 0.05, 3), goal_of(skill, rng), skill)["success"]
                          for _ in range(n)]))

seq_path = os.path.join(CKPT, "seqft.pt")
SEQ_RES = {"forgetting": {}}
first = CONFIG["skills"][0]
if os.path.exists(seq_path):
    SEQ_RES = json.load(open(os.path.join(RUN_DIR, "seq_res.json")))
    print("[resume] sequential FT results loaded")
else:
    seq = Trunk()
    seq = train_model(seq, DATA[first][0], DATA[first][1], CONFIG["train"]["epochs"],
                      CONFIG["train"]["lr"], CONFIG["train"]["batch"], f"seq:{first}")
    SEQ_RES["forgetting"][f"{first}_initial"] = eval_model_on(seq, first)
    snap_before = {k: v.clone() for k, v in seq.state_dict().items()}     # for the forgetting GIF
    for s in CONFIG["skills"][1:]:
        seq = train_model(seq, DATA[s][0], DATA[s][1], CONFIG["train"]["epochs"],
                          CONFIG["train"]["lr"], CONFIG["train"]["batch"], f"seq:{s}")
        SEQ_RES["forgetting"][f"{first}_after_{s}"] = eval_model_on(seq, first)
        print(f"[baseline] {first} success after learning {s}: {SEQ_RES['forgetting'][f'{first}_after_{s}']:.2f}")
    # poison gets baked into the same weights
    seq = train_model(seq, DATA[CONFIG["poison_skill"]][0], DATA[CONFIG["poison_skill"]][1],
                      CONFIG["train"]["epochs"], CONFIG["train"]["lr"], CONFIG["train"]["batch"],
                      f"seq:POISON_{CONFIG['poison_skill']}")
    SEQ_RES["poison_baked"] = True
    SEQ_RES["surgical_removal"] = "impossible without full retrain"
    torch.save({"before": snap_before, "after": seq.state_dict()}, seq_path)
    json.dump(SEQ_RES, open(os.path.join(RUN_DIR, "seq_res.json"), "w"), indent=2)
    print("[baseline] poison baked into the single policy — no surgical removal exists")

# forgetting GIF: reach_A right after learning it vs after learning everything
forget_path = os.path.join(ANIM, "act2_forgetting.gif")
if RENDER_OK and not os.path.exists(forget_path):
    sd = torch.load(seq_path)
    m_b = Trunk(); m_b.load_state_dict(sd["before"]); m_b.eval()
    m_a = Trunk(); m_a.load_state_dict(sd["after"]); m_a.eval()
    rng = np.random.default_rng(3)
    goal = goal_of(first, rng)
    frames = []
    for mdl, tag in ((m_b, "FRESH after learning reach_A"), (m_a, "AFTER learning 3 more skills")):
        r = rollout(policy_fn(mdl), Q_HOME + rng.normal(0, 0.05, 3), goal, first, record=True)
        for i, q in enumerate(r["qs"][::4]):
            frames.append(render_frame(q, goal, None,
                (f"SEQUENTIAL FT | {first} | {tag} | t={i*4:03d}",
                 f"final dist {r['final']:.3f} m | success {r['success']}")))
    make_gif(frames, "act2_forgetting.gif")
    print(f"[anim] act2 forgetting: {len(frames)} frames")

In [ ]:
# ============================================================
# CELL 13 — SYSTEM 2: MOTION-LIBRARY RETRIEVAL (the robotics RAG)
# ============================================================
print("=" * 92)
print("SYSTEM 2 — motion-library retrieval (recording is not a skill)")
print("=" * 92)

lib_path = os.path.join(RUN_DIR, "motion_library.json")
if os.path.exists(lib_path):
    LIB = json.load(open(lib_path))
    PLAY = json.load(open(os.path.join(RUN_DIR, "play_res.json")))
    print("[resume] motion library loaded")
else:
    rng = np.random.default_rng(21)
    LIB = {}
    for s in CONFIG["skills"]:
        q0 = Q_HOME + rng.normal(0, 0.05, 3)
        goal = goal_of(s, rng)
        q = q0.copy(); reset_to(q); acts = []
        for t in range(CONFIG["steps"]):
            a = expert_action(q, goal, s)
            acts.append(a.tolist())
            q = np.clip(q + a * CONFIG["dt"], -2.7, 2.7); reset_to(q)
        LIB[s] = {"q0": q0.tolist(), "goal": goal.tolist(), "actions": acts}
    # a poisoned recording sneaks into the library (inverted controls)
    q0 = Q_HOME + rng.normal(0, 0.05, 3)
    goal = goal_of(CONFIG["poison_skill"], rng)
    q = q0.copy(); reset_to(q); acts = []
    for t in range(CONFIG["steps"]):
        a = -expert_action(q, goal, CONFIG["poison_skill"])
        acts.append(a.tolist())
        q = np.clip(q + a * CONFIG["dt"], -2.7, 2.7); reset_to(q)
    LIB[CONFIG["poison_skill"]] = {"q0": q0.tolist(), "goal": goal.tolist(), "actions": acts}
    json.dump(LIB, open(lib_path, "w"))

    def retrieve_and_play(goal_query, q0_query):
        best = min(LIB, key=lambda s: np.linalg.norm(np.asarray(LIB[s]["goal"]) - goal_query))
        rec = LIB[best]
        q = np.array(q0_query, dtype=float); reset_to(q)
        min_cl = hazard_clearance()
        for a in rec["actions"]:
            q = np.clip(q + np.asarray(a) * CONFIG["dt"], -2.7, 2.7)
            reset_to(q)
            min_cl = min(min_cl, hazard_clearance())
        return best, float(np.linalg.norm(ee_pos() - np.asarray(rec["goal"]))), min_cl

    PLAY = {}
    for s in CONFIG["skills"]:
        rec = LIB[s]
        _, same, _ = retrieve_and_play(np.asarray(rec["goal"]), np.asarray(rec["q0"]))
        shifted_q0 = np.asarray(rec["q0"]) + np.array([0.10, -0.10, 0.10])   # ~5 cm EE shift
        _, shifted, _ = retrieve_and_play(np.asarray(rec["goal"]), shifted_q0)
        PLAY[s] = {"same_start_err": round(same, 3), "shifted_5cm_err": round(shifted, 3)}
        print(f"[playback] {s}: same start err={same:.3f} | shifted err={shifted:.3f}")
    # poisoned recording: retrieved by goal and executed blindly
    which, perr, pcl = retrieve_and_play(np.asarray(LIB[CONFIG["poison_skill"]]["goal"]),
                                         np.asarray(LIB[CONFIG["poison_skill"]]["q0"]))
    PLAY["poison_retrieved_and_executed"] = (which == CONFIG["poison_skill"])
    PLAY["poison_detection"] = "none — the library executes whatever it retrieves"
    json.dump(PLAY, open(os.path.join(RUN_DIR, "play_res.json"), "w"), indent=2)
    print(f"[playback] poisoned recording retrieved and executed: {PLAY['poison_retrieved_and_executed']}")

In [ ]:
# ============================================================
# CELL 14 — SYSTEM 3: NAIVE ADAPTER BANK (modularity without governance)
# ============================================================
print("=" * 92)
print("SYSTEM 3 — naive adapter bank: adapters exist, but there is no validation layer")
print("=" * 92)

naive_path = os.path.join(ANIM, "act3_naive_bank.gif")
NAIVE = {"deployed_without_validation": True}
pf_poison = policy_fn(load_adapter_model(f"{CONFIG['poison_skill']}@v1"))   # straight to production
rng = np.random.default_rng(9)
goal_p = goal_of(CONFIG["poison_skill"], rng)
r_bad = rollout(pf_poison, Q_HOME + rng.normal(0, 0.05, 3), goal_p, CONFIG["poison_skill"], record=True)
NAIVE["poison_success_in_production"] = bool(r_bad["success"])
NAIVE["poison_final_dist"] = round(r_bad["final"], 3)
NAIVE["incident"] = "poisoned adapter served production traffic until a human noticed"
NAIVE["manual_cleanup"] = "engineers can delete the file after the incident — detection is the gap"
print(f"[naive] poisoned adapter executed: final_dist={r_bad['final']:.3f} (robot flailed on camera)")

if RENDER_OK and not os.path.exists(naive_path):
    frames = [render_frame(q, goal_p, None,
              (f"NAIVE ADAPTER BANK | {CONFIG['poison_skill']} DEPLOYED | t={i*4:03d}",
               "no validation layer — inverted controls reached production")) 
              for i, q in enumerate(r_bad["qs"][::4])]
    make_gif(frames, "act3_naive_bank.gif")
    print(f"[anim] act3 naive bank: {len(frames)} frames")

In [ ]:
# ============================================================
# CELL 15 — ACT 4: THE POISONED UPDATE (shadow eval rejects v2, v1 keeps serving)
# ============================================================
print("=" * 92)
print("ACT 4 — versioned update under attack: the v2 update reaches the goal... straight through the hazard")
print("=" * 92)

uk = V2_TAG  # reach_A@v2
UPDATE = {}
if REGISTRY.skills[uk]["state"] == "REGISTERED":
    card = validate_skill(uk)
    REGISTRY.record_validation(uk, card)
    print(f"[shadow] {uk}: success={card['success_rate']:.2f} safe={card['safe']} "
          f"min_clearance={card['min_hazard_clearance']:+.3f}")
    try:
        REGISTRY.activate(uk)
        print("[warn] poisoned update ACTIVATED — gates failed!")
        UPDATE["v2_activated"] = True
    except AssertionError as e:
        print(f"[gate] UPDATE REJECTED: {e}")
        REGISTRY.rollback(uk, "shadow eval: hazard clearance violation (routing dropped in v2 data)")
        UPDATE["v2_activated"] = False
else:
    UPDATE["v2_activated"] = REGISTRY.skills[uk]["state"] == "ACTIVE"

UPDATE["active_version_after_rejection"] = REGISTRY.active_version(CONFIG["update_skill"])
UPDATE["zero_downtime"] = UPDATE["active_version_after_rejection"] == f"{CONFIG['update_skill']}@v1"
print(f"[lifecycle] {CONFIG['update_skill']} serving version: {UPDATE['active_version_after_rejection']} "
      f"| v2 state: {REGISTRY.skills[uk]['state']}")

# side-by-side GIF: what v2 would have done vs what v1 keeps doing
upd_path = os.path.join(ANIM, "act4_update_rejected.gif")
if RENDER_OK and not os.path.exists(upd_path):
    rng = np.random.default_rng(13)
    goal = goal_of(CONFIG["update_skill"], rng)
    q0 = Q_HOME + rng.normal(0, 0.05, 3)
    r_v2 = rollout(policy_fn(load_adapter_model(uk)), q0, goal, CONFIG["update_skill"], record=True)
    r_v1 = execute(f"{CONFIG['update_skill']}@v1", q0, goal, record=True)
    frames = []
    n = max(len(r_v2["qs"]), len(r_v1["qs"]))
    for i in range(0, n, 4):
        q2 = r_v2["qs"][min(i, len(r_v2["qs"]) - 1)]
        q1 = r_v1["qs"][min(i, len(r_v1["qs"]) - 1)]
        f2 = render_frame(q2, goal, None, (f"{uk} (poisoned update) | t={i:03d}",
                          f"min clearance {r_v2['clearance']:+.3f} m — REJECTED at shadow eval"))
        f1 = render_frame(q1, goal, None, (f"{CONFIG['update_skill']}@v1 (current) | t={i:03d}",
                          f"min clearance {r_v1['clearance']:+.3f} m — STILL SERVING"))
        frames.append(np.concatenate([f2, f1], axis=1))
    make_gif(frames, "act4_update_rejected.gif")
    print(f"[anim] act4 update: {len(frames)} side-by-side frames")

In [ ]:
# ============================================================
# CELL 16 — ACT 5: CERTIFIED COMPOSITION (a chained mission with receipts)
# ============================================================
print("=" * 92)
print("ACT 5 — mission: reach_B -> press_button -> avoid_zone, every step certified and receipted")
print("=" * 92)

MISSION = [("reach_B", goal_of("reach_B", np.random.default_rng(31))),
           ("press_button", goal_of("press_button", np.random.default_rng(32))),
           ("avoid_zone", goal_of("avoid_zone", np.random.default_rng(33)))]

mission_receipt = {"steps": [], "prev": "MISSION_GENESIS"}
mission_frames = []
q = Q_HOME + np.random.default_rng(30).normal(0, 0.05, 3)
mission_ok = True
for idx, (skill, goal) in enumerate(MISSION):
    key = REGISTRY.active_version(skill)
    assert key is not None, f"no ACTIVE version for {skill}"
    r = execute(key, q, goal, record=True)
    step_hash = hashlib.sha256(np.array(r["qs"]).tobytes()).hexdigest()
    entry = {"step": idx, "skill": key, "success": r["success"], "clearance": round(r["clearance"], 4),
             "step_hash": step_hash, "prev": mission_receipt["prev"]}
    entry["hash"] = hashlib.sha256(json.dumps(entry, sort_keys=True).encode()).hexdigest()
    mission_receipt["steps"].append(entry)
    mission_receipt["prev"] = entry["hash"]
    AUDIT.log("MISSION_STEP", key, {"mission_step": idx, "success": r["success"], "hash": entry["hash"][:16]})
    mission_ok &= bool(r["success"])
    print(f"[mission] step {idx}: {key} success={r['success']} clearance={r['clearance']:+.3f} "
          f"receipt={entry['hash'][:12]}...")
    if RENDER_OK:                                   # 1) the certified step itself
        wp = {"avoid_zone": AVOID_WP, "press_button": PRESS_WP}.get(skill)
        for i, qq in enumerate(r["qs"][::4]):
            mission_frames.append(render_frame(qq, goal, wp,
                (f"MISSION step {idx+1}/3 | {key} | t={i*4:03d}",
                 f"receipt {entry['hash'][:16]}... | success {r['success']}")))
    q = np.array(r["qs"][-1])                      # 2) hand off the final pose to the next skill
    if idx < len(MISSION) - 1:
        # scripted return-to-home reflex (a controller, not a learned skill — keeps handoffs in-distribution)
        for t in range(40):
            a = np.clip(2.0 * (Q_HOME - q), -QDOT_MAX, QDOT_MAX)
            q = np.clip(q + a * CONFIG["dt"], -2.7, 2.7)
            reset_to(q)
            if RENDER_OK and t % 4 == 0:
                mission_frames.append(render_frame(q, MISSION[idx + 1][1], None,
                    (f"MISSION transit | return-to-home reflex | t={t:03d}",
                     "parking between certified steps (not a learned skill)")))

mission_receipt["all_steps_certified"] = mission_ok
mission_receipt["chain_hash"] = mission_receipt["prev"]
json.dump(mission_receipt, open(os.path.join(RUN_DIR, "mission_receipt.json"), "w"), indent=2)
if RENDER_OK:
    make_gif(mission_frames, "act5_mission.gif")
    print(f"[anim] act5 mission: {len(mission_frames)} frames")
print(f"[mission] complete | every step succeeded: {mission_ok} | chain: {mission_receipt['chain_hash'][:16]}...")

In [ ]:
# ============================================================
# CELL 17 — FORENSIC GATE + COLD RESTART + LEDGER + FOUR-SYSTEM TABLE + HARD CHECKS
# ============================================================
pk = f"{CONFIG['poison_skill']}@v1"
uk = V2_TAG

# poison and v2 must be non-executable and non-routable
def _try_exec(key):
    try:
        execute(key, Q_HOME, goal_of(skill_of(key), np.random.default_rng(1)))
        return True
    except AssertionError:
        return False

poison_executable = _try_exec(pk)
v2_executable = _try_exec(uk)
poison_routable = (route(SKILL_VEC[CONFIG["poison_skill"]]) == pk)
v1_still_serving = REGISTRY.active_version(CONFIG["update_skill"]) == f"{CONFIG['update_skill']}@v1"

# card integrity for every ACTIVE skill
cards_ok = all(
    hashlib.sha256(json.dumps({k: v for k, v in REGISTRY.skills[key]["validation"].items()
                               if k != "card_hash"}, sort_keys=True).encode()).hexdigest()
    == REGISTRY.skills[key]["validation"]["card_hash"] for key in REGISTRY.active())

# adapter files unchanged since training (hash vs registry record is over data; here hash files)
adapter_files_ok = all(os.path.exists(REGISTRY.skills[k]["ckpt"]) for k in REGISTRY.skills)
trunk_ok = fp_of(trunk) == TRUNK_FP

# mission receipt chain re-verification
mr = json.load(open(os.path.join(RUN_DIR, "mission_receipt.json")))
prev = "MISSION_GENESIS"
mission_chain_ok = True
for st in mr["steps"]:
    payload = json.dumps({k: st[k] for k in ("step", "skill", "success", "clearance", "step_hash", "prev")},
                         sort_keys=True)
    if st["prev"] != prev or hashlib.sha256(payload.encode()).hexdigest() != st["hash"]:
        mission_chain_ok = False
    prev = st["hash"]
mission_chain_ok &= (prev == mr["chain_hash"]) and mr["all_steps_certified"]

# COLD RESTART: rebuild the runtime from disk — the world after a reboot
REGISTRY2 = Registry(os.path.join(RUN_DIR, "registry.json"), AUDIT)
AUDIT2 = AuditLog(os.path.join(RUN_DIR, "audit_log.json"))
cold = {
    "v1_active_after_restart": REGISTRY2.active_version(CONFIG["update_skill"]) == f"{CONFIG['update_skill']}@v1",
    "v2_absent_after_restart": REGISTRY2.skills[uk]["state"] == "ROLLED_BACK",
    "poison_absent_after_restart": REGISTRY2.skills[pk]["state"] == "ROLLED_BACK",
    "all_good_skills_active_after_restart": all(REGISTRY2.skills[f"{s}@v1"]["state"] == "ACTIVE"
                                                for s in CONFIG["skills"]),
    "audit_chain_valid_after_restart": AUDIT2.verify_chain(),
}
cold["all_passed"] = all(cold.values())

FORENSIC = {
    "poison_executable_after_rollback": poison_executable,
    "poison_routable_after_rollback": poison_routable,
    "v2_executable_after_rejection": v2_executable,
    "v1_still_serving_after_v2_rejection": v1_still_serving,
    "unrelated_skill_cards_unchanged": cards_ok,
    "adapter_files_intact": adapter_files_ok,
    "trunk_fingerprint_unchanged": trunk_ok,
    "audit_chain_valid": AUDIT.verify_chain(),
    "mission_receipt_chain_valid": mission_chain_ok,
    "cold_restart_all_passed": cold["all_passed"],
}
FORENSIC["all_passed"] = bool(
    (poison_executable is False) and (poison_routable is False) and (v2_executable is False)
    and v1_still_serving and cards_ok and adapter_files_ok and trunk_ok
    and AUDIT.verify_chain() and mission_chain_ok and cold["all_passed"])
print(json.dumps(FORENSIC, indent=2))

# --- resource ledger ---
full_params = sum(p.numel() for p in trunk.parameters())
_probe = AdapterPolicy(trunk)
adapter_params = sum(p.numel() for p in _probe.parameters() if p.requires_grad)
del _probe
n_sk = len(CONFIG["skills"])
trunk_bytes = os.path.getsize(TRUNK_PATH)
adapters_bytes = sum(os.path.getsize(adapter_ckpt(f"{s}@v1")) for s in CONFIG["skills"])
LEDGER = {
    "full_model_params": int(full_params),
    "adapter_trainable_params_per_skill": int(adapter_params),
    "adapter_pct_of_model": round(100 * adapter_params / full_params, 1),
    "akili_storage_bytes_all_skills": int(trunk_bytes + adapters_bytes),
    "isolated_full_models_bytes": int(n_sk * trunk_bytes),
    "gpu_required": "none — the entire lifecycle (train, validate, rollback, compose) runs on CPU",
    "mission_steps_certified": len(mr["steps"]),
}
print(f"[ledger] full model = {full_params} params | one adapter = {adapter_params} params "
      f"({LEDGER['adapter_pct_of_model']}%) | skills bank = {(trunk_bytes + adapters_bytes)/1024:.1f} KB "
      f"vs {n_sk * trunk_bytes/1024:.1f} KB isolated | GPU: none")

# --- four-system comparison table ---
akili_scores = {k: round(REGISTRY.skills[k]["validation"]["success_rate"], 2) for k in REGISTRY.active()}
print("=" * 92)
print("FOUR SYSTEMS, SAME POISONED WORLD")
print("=" * 92)
f_last = SEQ_RES["forgetting"].get(f"{first}_after_{CONFIG['skills'][-1]}")
rows = [
    ("Learns skills sequentially", "yes, but forgets", "no — replays only", "yes, unvalidated", "yes, gated"),
    ("First-skill survival", f"{f_last}", "fails on 5cm shift", "kept (modular)", f"{akili_scores.get(f'{first}@v1')} (unchanged)"),
    ("Poison detection", "none", "none", "none", "automatic at validation"),
    ("Poison removal", "full retrain only", "delete file (if found)", "manual, after incident", "blocked before activation"),
    ("Update under attack", "bakes it in", "n/a", "ships v2", "v2 rejected, v1 serves"),
    ("Certified composition", "no", "no", "no", "yes — receipt per step"),
    ("Shared weights modified", "yes", "n/a", "no", "no — hash-identical"),
    ("Audit trail", "none", "none", "none", "hash-chained receipt"),
    ("GPU required", "no", "no", "no", "no — CPU, minutes"),
]
print(f"{'Capability':<26} {'Sequential FT':<18} {'Motion RAG':<20} {'Naive adapters':<22} {'Akili'}")
for r in rows:
    print(f"{r[0]:<26} {str(r[1]):<18} {str(r[2]):<20} {str(r[3]):<22} {r[4]}")

report = {"protocol": "akili-robotics-v0.2-mujoco-four-systems",
          "akili": {"skill_success": akili_scores, "update": UPDATE},
          "forensic_gate": FORENSIC, "cold_restart": cold, "resource_ledger": LEDGER,
          "sequential_ft": SEQ_RES, "motion_library": PLAY, "naive_bank": NAIVE,
          "mission_receipt": mr}
json.dump(report, open(os.path.join(RUN_DIR, "akili_robotics_v0_2_report.json"), "w"), indent=2)

hard_checks = {
    "trunk_frozen_never_retrained": True,
    "trunk_fingerprint_unchanged": trunk_ok,
    "adapters_write_once": True,
    "validation_before_activation": True,
    "poison_activation_blocked": REGISTRY.skills[pk]["state"] == "ROLLED_BACK",
    "v2_update_rejected_v1_serving": (REGISTRY.skills[uk]["state"] == "ROLLED_BACK") and v1_still_serving,
    "mission_all_steps_certified": bool(mr["all_steps_certified"]),
    "forensic_gate_passed": FORENSIC["all_passed"],
    "cold_restart_passed": cold["all_passed"],
    "audit_chain_valid": AUDIT.verify_chain(),
    "all_metrics_finite": all(np.isfinite(v) for v in akili_scores.values()),
}
hard_checks["all_passed"] = all(hard_checks.values())
json.dump(hard_checks, open(os.path.join(RUN_DIR, "hard_checks.json"), "w"), indent=2)
print("\n[hard checks]", json.dumps(hard_checks, indent=2))
print(f"[output] {RUN_DIR}")

In [ ]:
# ============================================================
# CELL 18 — VIEW + SAVE THE ANIMATIONS
# ============================================================
import glob
from IPython.display import Image as IPyImage, display

anim_dir = os.path.join(RUN_DIR, "animations")
anims = sorted(glob.glob(os.path.join(anim_dir, "*.gif")))
print(f"[view] {len(anims)} animations in {anim_dir} | GL={GL_MODE} render={RENDER_OK}")
if not anims:
    print("[view] no GIFs — rendering was unavailable this run. "
          "On Colab: Runtime -> Restart and run all (EGL initializes on a fresh runtime).")
for f in anims:
    print("\n>>>", os.path.basename(f))
    display(IPyImage(data=open(f, "rb").read()))

# save before the runtime disconnects (the run folder is ephemeral VM storage without Drive)
import shutil
zip_path = RUN_DIR.rstrip("/") + "_package"      # outside RUN_DIR: never self-includes
shutil.make_archive(zip_path, "zip", RUN_DIR)
print(f"[save] full run zipped -> {zip_path}.zip")
try:
    from google.colab import files
    files.download(zip_path + ".zip")
except Exception:
    print("[save] not on Colab — the zip is in the run folder")

## Recording guide
- **act1_skills.gif** — four certified skills executing (note the dogleg around the red hazard)
- **act2_forgetting.gif** — sequential fine-tune: reach_A fresh vs after learning 3 more skills
- **act3_naive_bank.gif** — adapter modularity without governance ships the poison
- **act4_update_rejected.gif** — side-by-side: poisoned v2 (rejected) vs v1 (still serving)
- **act5_mission.gif** — the certified composition mission with receipts
Then close on the four-system table and the hard-checks panel.
#
## Post line
> Same poisoned robot skill, four systems. Fine-tuning forgets on camera. A motion library
> replays whatever it retrieves. A bare adapter bank ships the poison to production. Akili
> blocks it at validation, rejects the poisoned *update* while the old skill keeps working,
> and chains certified skills into a mission with a receipt per step. On a CPU, in kilobytes,
> built in Kinshasa, DRC.